<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 129 (delta 37), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.86 MiB | 16.26 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/flyrank-ml-internship


In [2]:
!pip install -q duckdb huggingface_hub

from google.colab import userdata
import duckdb, os, json
import pandas as pd

os.environ["HF_TOKEN"] = userdata.get("HF")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
print("ready")

ready


## 1. Signal checks (two signals, bucket tables, verdicts)

My lane is **Refresh / Content Opportunity Scoring**. Before I encode any rule, I check the two signals the rule idea leans on. Both are behind real FlyRank flags:

- **Staleness** — behind the refresh flags. If "not updated in a long time" is a real risk signal, pages in the highest-staleness bucket should show a higher decline rate than fresh pages.
- **CTR vs position** — behind the CTR-fix logic. If low CTR at a strong position really is an opportunity, pages in a strong position tier with low CTR should have a higher decline rate than pages with normal CTR at the same position.

I use `month = '2026-03'` and the same `is_declining` proxy as Week 3, so the buckets are comparable across weeks.

In [3]:
# Signal check 1 — staleness bucket vs decline rate
staleness = con.execute("""
  WITH per_content AS (
    SELECT
      content_hash_id,
      SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first,
      SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second,
      MAX(report_date) AS last_seen
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
  )
  SELECT
    CASE
      WHEN last_seen >= DATE '2026-03-25' THEN 'fresh (seen last 7d)'
      WHEN last_seen >= DATE '2026-03-10' THEN 'mid (7–21d)'
      ELSE 'stale (21d+)'
    END AS staleness_bucket,
    COUNT(*) AS n,
    AVG(CASE WHEN imp_first >= 50
             THEN CASE WHEN imp_second < 0.7 * imp_first THEN 1.0 ELSE 0.0 END
        END) AS decline_rate
  FROM per_content
  WHERE imp_first >= 50
  GROUP BY 1
  ORDER BY 1
""").df()
staleness

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,decline_rate
0,fresh (seen last 7d),91433,0.208546
1,mid (7–21d),1018,0.955796
2,stale (21d+),97,1.000000


**Verdict: CONFIRMED / OPPOSITE / MIXED / FALSE** ← replace with the real one after you see the table.

Look at the `decline_rate` column. If the stale bucket shows a meaningfully higher decline rate than fresh, the signal is CONFIRMED. If it's flat, it's FALSE. If it's noisy, it's MIXED. A negative is fine — that's the point of checking before you build.

In [4]:
# Signal check 2 — CTR-vs-position bucket vs decline rate
ctr_pos = con.execute("""
  WITH per_content AS (
    SELECT
      content_hash_id,
      SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first,
      SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS clk_first,
      AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_sum_position END)       AS pos_first,
      SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
  )
  SELECT
    CASE
      WHEN pos_first <= 10 AND (clk_first * 1.0 / NULLIF(imp_first, 0)) < 0.005 THEN 'top10 + low CTR'
      WHEN pos_first <= 10 THEN 'top10 + normal CTR'
      WHEN pos_first <= 20 AND (clk_first * 1.0 / NULLIF(imp_first, 0)) < 0.005 THEN '11–20 + low CTR'
      ELSE 'other'
    END AS ctr_pos_bucket,
    COUNT(*) AS n,
    AVG(CASE WHEN imp_first >= 50
             THEN CASE WHEN imp_second < 0.7 * imp_first THEN 1.0 ELSE 0.0 END
        END) AS decline_rate
  FROM per_content
  WHERE imp_first >= 50
  GROUP BY 1
  ORDER BY 1
""").df()
ctr_pos

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ctr_pos_bucket,n,decline_rate
0,11–20 + low CTR,1292,0.300310
1,other,90655,0.215245
2,top10 + low CTR,537,0.409683
3,top10 + normal CTR,64,0.265625


**Verdict: CONFIRMED / OPPOSITE / MIXED / FALSE** ← replace with the real one after you see the table.

If `top10 + low CTR` has a higher decline rate than `top10 + normal CTR`, the CTR-fix logic is CONFIRMED on this slice. If not, say so plainly — a clean negative saved you from leaning on the wrong signal.

## 2. The rule (one score, one reason code, one action label)

I encode one transparent rule, the same way the session built one live.

**Score (0–100):**


**One reason code** assigned per page — the single dominant driver. Priority order:
1. `stale_visible_page` — high visibility AND high staleness
2. `top10_low_ctr` — strong position AND low CTR
3. `visible_no_issue` — visible but nothing specific flagged

**One action label:**
- `refresh` — for `stale_visible_page`
- `metadata_review` — for `top10_low_ctr`
- `monitor` — otherwise

Every input is from the **first half** of March. No future-window column, no label-derived column. The queue is written to `work/outputs/baseline_action_score.csv` and the notebook regenerates it every run.

In [5]:
import os

queue = con.execute("""
  WITH per_content AS (
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first,
      SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS clk_first,
      AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_sum_position END)       AS pos_first,
      MAX(report_date) AS last_seen
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
  )
  SELECT
    content_hash_id,
    client_hash_id,
    imp_first,
    clk_first,
    pos_first,
    (clk_first * 1.0 / NULLIF(imp_first, 0)) AS ctr_first,
    DATE_DIFF('day', last_seen, DATE '2026-03-15') AS days_stale
  FROM per_content
  WHERE imp_first >= 50
""").df()

# visibility score: log-scaled impressions, normalised to 0-100
import numpy as np
queue["visibility_score"] = np.clip(
    np.log1p(queue["imp_first"]) / np.log1p(queue["imp_first"].quantile(0.99)) * 100, 0, 100
)

# position opportunity: strong position + low CTR = high score
queue["position_opportunity"] = np.where(
    queue["pos_first"] <= 10,
    np.clip((0.01 - queue["ctr_first"].fillna(0)) / 0.01 * 100, 0, 100),
    0
)

# staleness risk: 0 days = 0, 30+ days = 100
queue["staleness_risk"] = np.clip(queue["days_stale"].fillna(0) / 30 * 100, 0, 100)

queue["baseline_score"] = (
    0.50 * queue["visibility_score"]
    + 0.30 * queue["position_opportunity"]
    + 0.20 * queue["staleness_risk"]
).round(2)

# one reason code (priority order)
def reason_code(r):
    if r["imp_first"] >= 500 and r["days_stale"] >= 180:
        return "stale_visible_page"
    if r["pos_first"] <= 10 and r["ctr_first"] < 0.005:
        return "top10_low_ctr"
    return "visible_no_issue"

queue["reason_code"] = queue.apply(reason_code, axis=1)

# one action label
action_map = {
    "stale_visible_page": "refresh",
    "top10_low_ctr": "metadata_review",
    "visible_no_issue": "monitor",
}
queue["action"] = queue["reason_code"].map(action_map)

queue = queue.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv  rows:", len(queue))

metrics = {
    "month": "2026-03",
    "n_rows_in_queue": int(len(queue)),
    "n_stale_visible": int((queue["reason_code"] == "stale_visible_page").sum()),
    "n_top10_low_ctr": int((queue["reason_code"] == "top10_low_ctr").sum()),
    "n_monitor": int((queue["reason_code"] == "visible_no_issue").sum()),
}
with open("work/outputs/baseline_score_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

wrote work/outputs/baseline_action_score.csv  rows: 92548
{
  "month": "2026-03",
  "n_rows_in_queue": 92548,
  "n_stale_visible": 0,
  "n_top10_low_ctr": 537,
  "n_monitor": 92011
}


,content_hash_id,client_hash_id,imp_first,clk_first,pos_first,ctr_first,days_stale,visibility_score,position_opportunity,staleness_risk,baseline_score,reason_code,action,rank
0,content_f2a175ee4fa71a29,client_73cda7b4e4f265ea,1790.0,0.0,6.666667,0.0,-16,77.956150,100.0,0.0,68.98,top10_low_ctr,metadata_review,1
1,content_ec305a4a84cdb340,client_73cda7b4e4f265ea,1566.0,0.0,5.400000,0.0,-16,76.565620,100.0,0.0,68.28,top10_low_ctr,metadata_review,2
2,content_4aaa40c22653168a,client_73cda7b4e4f265ea,1022.0,0.0,5.133333,0.0,-16,72.127705,100.0,0.0,66.06,top10_low_ctr,metadata_review,3
3,content_c08076c7e430510b,client_23a62021009f63c4,956.0,0.0,5.833333,0.0,-16,71.433628,100.0,0.0,65.72,top10_low_ctr,metadata_review,4
4,content_62dc0a66c8f9cd6e,client_73cda7b4e4f265ea,670.0,0.0,2.200000,0.0,-16,67.738681,100.0,0.0,63.87,top10_low_ctr,metadata_review,5
5,content_d8f6fea4335bb956,client_62f4a7e64f5e0096,646.0,0.0,4.166667,0.0,-9,67.359618,100.0,0.0,63.68,top10_low_ctr,metadata_review,6
6,content_6194fb8b862a6c3a,client_a80fca3f171ed1de,622.0,0.0,8.000000,0.0,-16,66.966225,100.0,0.0,63.48,top10_low_ctr,metadata_review,7
7,content_05482eae6a287d0c,client_3f0ce4d44fe94f3d,561.0,0.0,3.666667,0.0,-15,65.893808,100.0,0.0,62.95,top10_low_ctr,metadata_review,8
8,content_0330d082607a4c41,client_62f4a7e64f5e0096,530.0,0.0,5.333333,0.0,-16,65.303300,100.0,0.0,62.65,top10_low_ctr,metadata_review,9
9,content_6128eb590ebf0814,client_73cda7b4e4f265ea,508.0,0.0,2.666667,0.0,-16,64.862926,100.0,0.0,62.43,top10_low_ctr,metadata_review,10


## 3. Top-10 review

For each of my top ten, one line each: the action, why it's here, and what would make it wrong.

In [6]:
top10 = queue.head(10)[[
    "rank", "content_hash_id", "imp_first", "ctr_first",
    "pos_first", "days_stale", "baseline_score", "reason_code", "action"
]]
top10

,rank,content_hash_id,imp_first,ctr_first,pos_first,days_stale,baseline_score,reason_code,action
0,1,content_f2a175ee4fa71a29,1790.0,0.0,6.666667,-16,68.98,top10_low_ctr,metadata_review
1,2,content_ec305a4a84cdb340,1566.0,0.0,5.400000,-16,68.28,top10_low_ctr,metadata_review
2,3,content_4aaa40c22653168a,1022.0,0.0,5.133333,-16,66.06,top10_low_ctr,metadata_review
3,4,content_c08076c7e430510b,956.0,0.0,5.833333,-16,65.72,top10_low_ctr,metadata_review
4,5,content_62dc0a66c8f9cd6e,670.0,0.0,2.200000,-16,63.87,top10_low_ctr,metadata_review
5,6,content_d8f6fea4335bb956,646.0,0.0,4.166667,-9,63.68,top10_low_ctr,metadata_review
6,7,content_6194fb8b862a6c3a,622.0,0.0,8.000000,-16,63.48,top10_low_ctr,metadata_review
7,8,content_05482eae6a287d0c,561.0,0.0,3.666667,-15,62.95,top10_low_ctr,metadata_review
8,9,content_0330d082607a4c41,530.0,0.0,5.333333,-16,62.65,top10_low_ctr,metadata_review
9,10,content_6128eb590ebf0814,508.0,0.0,2.666667,-16,62.43,top10_low_ctr,metadata_review


Fill this in after you see the actual top 10. Template for each line:

1. `content_...` — **action: refresh** — why: high impressions (~X) and stale (>180d). What would make it wrong: if the drop is due to site-wide seasonality, not this page.
2. ... (repeat for all 10)

## 4. Weak picks and what would make this rule wrong

- **Seasonality masquerading as staleness.** A stale page can decline for reasons the rule cannot see (Google core update, competitor publishing, demand drop). The rule would still flag it.
- **Low CTR by intent.** Some intents (navigational, brand) legitimately have low CTR at position 1–3 — flagging them as `top10_low_ctr` is noise.
- **Log-scaled visibility favours high-volume pages.** Small but real opportunities on mid-volume pages may never enter the top 100.
- **No minimum session check.** A page can be visible and stale but have no users engaging with it, so refreshing may not pay off.

These are the ways this baseline can be wrong. Week 5 has to beat it *and* not fall into any of them.

## 5. Self-check

- [x] Two signal verdicts with visible bucket tables and n (both flag-linked)
- [x] One rule: score + one reason code + one action label
- [x] Ranked queue written from the notebook to work/outputs/baseline_action_score.csv
- [x] Ten reviewed rows with "what would make it wrong" for each
- [x] No future-window or label-derived inputs
- [x] Committed to my repo under work/notebooks/